# 39 - Silver labels at full scale, using the existing gold set to distill a free classifier

Judging every candidate with an LLM does not scale affordably (a true gold standard across all 101 queries would cost roughly $22-28,000 depending on pooling depth, see the cost breakdown from the previous conversation). Instead, this trains a final model on the 419 gold-labeled candidates from notebooks 34-38, then applies it (zero additional API cost) to all 173,262 (query, candidate) pairs already scored in the original fusion ranker's feature table, covering all 101 queries.

This produces two explicitly separate tiers, gold vs silver, a standard distinction in weak/distant supervision:
- **Gold** (419 candidates): LLM-judged, human-tie-broken, expensive and trustworthy.
- **Silver** (the remaining ~172,845): model-predicted, free, less trustworthy, validated by spot-checking a small sample against real LLM judgments.

**Do not report silver labels as if they were gold.** State the tier explicitly wherever these labels are used.

In [6]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

OUTPUT_DIR = Path("result/39_silver_labels_full_scale")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2  # same strict "highly relevant" definition used in notebook 38

gold = pd.read_json("result/35_llm_judge_ensemble/final_gold_labels.json")
features = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")

feature_cols = ['score_minilm', 'score_linq', 'score_gte', 'score_bm25',
                'invrank_minilm', 'invrank_linq', 'invrank_gte', 'invrank_bm25',
                'reranker_score', 'n_channels']

labeled = gold.merge(features, on=["query_id", "domain"], how="inner")
labeled["relevant"] = (labeled["gold_label"] >= RELEVANT_THRESHOLD).astype(int)
print(f"Gold candidates matched to features: {len(labeled)}/{len(gold)}")
print(f"Relevant (gold_label >= {RELEVANT_THRESHOLD}): {labeled['relevant'].sum()}/{len(labeled)}")

Gold candidates matched to features: 417/419
Relevant (gold_label >= 2): 291/417


In [7]:
# Train the FINAL model on all available gold labels (no CV split here -- CV in notebook 38
# was for honestly measuring quality; for the deployed silver-labeling model we want every
# gold label used to fit the best possible final classifier).
X_train = labeled[feature_cols].values
y_train = labeled["relevant"].values

final_gbdt = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight="balanced", random_state=0)
final_gbdt.fit(X_train, y_train)

final_logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
final_logreg.fit(X_train, y_train)

print("Final model trained on all", len(labeled), "gold-labeled candidates.")
print("Feature weights (logistic regression, for reference):")
for name, coef in sorted(zip(feature_cols, final_logreg.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"    {name:<16} {coef:+.3f}")

Final model trained on all 417 gold-labeled candidates.
Feature weights (logistic regression, for reference):
    score_linq       +2.410
    invrank_bm25     -1.278
    invrank_minilm   -1.007
    score_gte        +0.902
    n_channels       -0.490
    invrank_linq     +0.410
    invrank_gte      +0.403
    reranker_score   +0.309
    score_bm25       -0.136
    score_minilm     -0.098


In [8]:
# Predict for every one of the 173,262 candidates across all 101 queries -- zero additional API cost.
X_all = features[feature_cols].values
features["silver_prob_relevant"] = final_gbdt.predict_proba(X_all)[:, 1]
features["silver_label"] = (features["silver_prob_relevant"] >= 0.5).astype(int)

# Mark which rows already have a real gold label -- don't let the silver prediction
# shadow or contradict a label you already paid an LLM to produce.
gold_keys = set(zip(labeled["query_id"], labeled["domain"]))
features["tier"] = ["gold" if (q, d) in gold_keys else "silver" for q, d in zip(features["query_id"], features["domain"])]

silver_path = OUTPUT_DIR / "silver_labels.json"
features.to_json(silver_path, orient="records", indent=2)

print(f"Total candidates: {len(features)}")
print(features["tier"].value_counts())
print(f"Saved -> {silver_path}")
print()
print("Silver-only predicted-relevant rate:")
print(features[features["tier"] == "silver"]["silver_label"].value_counts(normalize=True))

Total candidates: 173262
tier
silver    172845
gold         417
Name: count, dtype: int64
Saved -> result/39_silver_labels_full_scale/silver_labels.json

Silver-only predicted-relevant rate:
silver_label
1    0.753641
0    0.246359
Name: proportion, dtype: float64


## Spot-check the silver labels against real LLM judgments

Sample a small random set of silver-only candidates and judge them for real, with the stronger models (Claude Sonnet 5 + GPT-5.4), to see how well the free model-predicted labels actually hold up. This costs under $1, not the $22-28,000 a full re-judging pass would cost.

**Note on `gpt-5.4`**: verify this exact model string works for your OpenAI account before relying on it -- if it 404s, check OpenAI's current model list and swap in whatever the equivalent current mainstream-tier model ID is.

In [9]:
import os, time, requests
from dotenv import load_dotenv

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

SPOT_CHECK_N = 150

# Text lookups: domain -> name/summary (global, dedup by domain), query_id -> query text.
corpus = pd.read_csv("dataset/company_corpus.csv")
domain_lookup = corpus.drop_duplicates(subset="domain").set_index("domain")
query_lookup = corpus[["query_id", "query"]].drop_duplicates().set_index("query_id")["query"]

silver_only = features[features["tier"] == "silver"].copy()
silver_only = silver_only[silver_only["domain"].isin(domain_lookup.index)]  # keep only rows we can attach text to
spot_sample = silver_only.sample(n=SPOT_CHECK_N, random_state=42).reset_index(drop=True)
spot_sample["query"] = spot_sample["query_id"].map(query_lookup)
spot_sample["name"] = spot_sample["domain"].map(domain_lookup["name"])
spot_sample["summary"] = spot_sample["domain"].map(domain_lookup["summary"])
spot_sample["country"] = spot_sample["domain"].map(domain_lookup["country"])
print(f"Spot-check sample: {len(spot_sample)} silver-labeled candidates")

Spot-check sample: 150 silver-labeled candidates


In [10]:
JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Country: {country}
Summary: {summary}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant (a strong, direct match for the query)
1 = partially relevant (related but not a strong direct match)
0 = not relevant

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def build_prompt(row):
    return JUDGE_PROMPT_TEMPLATE.format(query=row["query"], name=row["name"], country=row["country"], summary=row["summary"])


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"


def judge_openai_gpt54(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-5.4", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude_sonnet5(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={
            "model": "claude-sonnet-5", "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=60,
    )
    resp.raise_for_status()
    # Sonnet 5 runs adaptive thinking by default, so content[0] may be a "thinking" block,
    # not "text" -- find the actual text block instead of assuming it's first.
    content_blocks = resp.json()["content"]
    text_block = next((b["text"] for b in content_blocks if b.get("type") == "text"), None)
    if text_block is None:
        return None, f"NO TEXT BLOCK: {content_blocks}"
    return parse_judge_reply(text_block)


spot_check_cache_path = OUTPUT_DIR / "spot_check_cache.json"
spot_results = json.load(open(spot_check_cache_path)) if spot_check_cache_path.exists() else {}

for i, row in spot_sample.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = spot_results.get(key, {})
    prompt = build_prompt(row)

    if "gpt54" not in entry:
        try:
            label, reason = judge_openai_gpt54(prompt)
            entry["gpt54"] = {"label": label, "reason": reason}
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [gpt-5.4] error on {row['domain']}: {e}")
    if "sonnet5" not in entry:
        try:
            label, reason = judge_claude_sonnet5(prompt)
            entry["sonnet5"] = {"label": label, "reason": reason}
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [sonnet5] error on {row['domain']}: {e}")

    spot_results[key] = entry
    json.dump(spot_results, open(spot_check_cache_path, "w"), indent=2, default=str)  # save after every row

    if (i + 1) % 25 == 0 or (i + 1) == len(spot_sample):
        print(f"  {i+1}/{len(spot_sample)} spot-checked")
    time.sleep(0.2)

print("Done.")

  25/150 spot-checked
  50/150 spot-checked
  75/150 spot-checked
  100/150 spot-checked
  125/150 spot-checked
  150/150 spot-checked
Done.


In [11]:
rows = []
for i, row in spot_sample.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = spot_results.get(key, {})
    if "gpt54" not in entry or "sonnet5" not in entry:
        continue
    gpt_label, sonnet_label = entry["gpt54"]["label"], entry["sonnet5"]["label"]
    if gpt_label is None or sonnet_label is None:
        continue
    # Treat unanimous agreement between the two strong judges as the "true" spot-check label;
    # disagreements are excluded here rather than guessed at (same principle as notebook 35).
    if gpt_label != sonnet_label:
        continue
    true_relevant = int(gpt_label >= RELEVANT_THRESHOLD)
    rows.append({"domain": row["domain"], "silver_label": row["silver_label"], "true_relevant": true_relevant})

spot_eval = pd.DataFrame(rows)
accuracy = (spot_eval["silver_label"] == spot_eval["true_relevant"]).mean() if len(spot_eval) else float("nan")
print(f"Spot-checked with unanimous strong-judge agreement: {len(spot_eval)}/{len(spot_sample)}")
print(f"Silver label accuracy against that: {accuracy:.1%}" if len(spot_eval) else "No unanimous pairs yet")

Spot-checked with unanimous strong-judge agreement: 132/150
Silver label accuracy against that: 65.9%
